In [1]:
#!/usr/bin/env python3
"""The first code cell of every notebook a job may run — copied byte for byte.

One question, answered once: WHERE IS THE REPOSITORY THIS NOTEBOOK RUNS
AGAINST. Every later cell reads `ROOT` and none of them asks again.

Opened by a person on their own machine, this answers it exactly the way
these notebooks always have. A notebook lives at `<repo>/notebooks/`,
so the repository is two directories above the working directory. Nothing was
handed over, nothing is checked, and the behaviour is unchanged.

Started by a runner on a remote worker, it does not answer it by looking
around. The runner exports the directory it cloned the pinned commit into and
the commit it pinned; this cell reads both, and PROVES the checkout is at that
commit before returning it.

Guessing is the whole reason this cell exists. Under the runner the kernel's
working directory is the runner's own and the clone sits one level inside it,
so two directories up is two levels ABOVE the working directory — a directory
that EXISTS on any worker. The path resolves, the insert succeeds, and the run
dies later with a missing module naming a package, never with the wrong root.
The one fact worth having is the one the failure never mentions.

Three refusals, and each one is a refusal rather than a fallback because this
is the path that spends metered quota:

- **Half a handoff** — one variable present and the other missing. Something
  built this environment and got it half right; that is precisely the state a
  fallback cannot tell apart from a laptop, and the fallback resolves a
  directory that exists everywhere.
- **A root that is not a checkout** — the declared directory is missing, or
  holds no readable `HEAD`. There is nothing to compare against, and an
  unproven root is what this cell was written to stop being acceptable.
- **A checkout at a different commit** — the declared commit and the one on
  disk disagree. A job that runs the wrong commit RUNS: it returns numbers
  shaped exactly like the right ones, and nothing downstream can tell.

Absent means LOCAL. It never means "work it out".

Importable and independently testable, the way the runner's own two cells
are: every function is pure and takes its environment and its working
directory as arguments, and only the last line — the binding a notebook cell
exists to perform — reads the real ones.
"""
import os
from pathlib import Path

#: The directory a runner cloned the pinned commit into. Forge-owned and
#: deliberately generic: this is the contract between a runner and the
#: notebook it starts, not a name borrowed from any one repository.
CLONE_ROOT_ENV = "FORGE_CLONE_ROOT"

#: The commit that clone was pinned to, exported beside it so this cell can
#: check rather than trust. A root on its own would only move the guess one
#: step: a directory handed over is still a directory nobody proved.
CLONE_COMMIT_ENV = "FORGE_CLONE_COMMIT"

#: How far above a notebook's own directory the repository sits when nobody
#: hands anything over. This repository uses the flat layout
#: `<repo>/notebooks` -> `<repo>`, so the depth is 0.
LOCAL_ROOT_DEPTH = 0


def head_commit(root):
    """The commit `root`'s `HEAD` names, read straight out of `.git`.

    No subprocess, deliberately. A notebook cell that shells out needs a git
    binary on a worker's PATH that nothing here declared, and every checker
    that reads these notebooks then has to decide whether running the cell is
    safe — a cell that binds the repository is the last one that may be
    skipped for that reason.

    A pinned checkout is detached: the runner fetches a commit and checks out
    what it fetched, never a branch, so `HEAD` holds the raw commit and this
    is a one-line read. A symbolic `HEAD` is resolved anyway — through the
    loose ref and then `packed-refs` — so pointing these variables at an
    ordinary checkout gets an answer instead of a refusal that would be about
    the file format rather than about the commit.

    Returns `None` when there is nothing to read. The caller turns that into
    a refusal; this function never decides.
    """
    git = Path(root) / ".git"
    if git.is_file():
        pointer = git.read_text(encoding="utf-8").strip()
        if not pointer.startswith("gitdir:"):
            return None
        git = Path(root) / pointer[len("gitdir:"):].strip()
    if not git.is_dir():
        return None
    head = git / "HEAD"
    if not head.is_file():
        return None
    text = head.read_text(encoding="utf-8").strip()
    if not text.startswith("ref:"):
        return text or None
    ref = text[len("ref:"):].strip()
    loose = git / ref
    if loose.is_file():
        return loose.read_text(encoding="utf-8").strip() or None
    packed = git / "packed-refs"
    if packed.is_file():
        for line in packed.read_text(encoding="utf-8").splitlines():
            if line.startswith("#") or line.startswith("^"):
                continue
            fields = line.split()
            if len(fields) == 2 and fields[1] == ref:
                return fields[0]
    return None


def resolve_repository_root(environ=None, cwd=None, head_reader=head_commit):
    """The repository this notebook runs against, or a refusal saying why.

    `environ` and `cwd` are arguments so the whole decision can be driven
    without a kernel, a clone or a worker; the binding at the bottom of this
    cell passes the real ones.
    """
    environ = os.environ if environ is None else environ
    here = Path.cwd() if cwd is None else Path(cwd)
    declared_root = environ.get(CLONE_ROOT_ENV)
    declared_commit = environ.get(CLONE_COMMIT_ENV)

    if not declared_root and not declared_commit:
        return here.parents[LOCAL_ROOT_DEPTH]

    if not declared_root or not declared_commit:
        raise RuntimeError(
            "half a handoff: {0}={1!r} and {2}={3!r}. Both name the pinned "
            "checkout this notebook must run against, and one without the "
            "other is an environment somebody built and got half right. "
            "Falling back to the local layout here would resolve a directory "
            "that exists on any machine and is not this repository.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))

    root = Path(declared_root)
    found = head_reader(root) if root.is_dir() else None
    if found is None:
        raise RuntimeError(
            "{0}={1!r} is not a readable checkout: no commit could be read "
            "from its HEAD. {2} declares {3!r}, and there is nothing here to "
            "check it against.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))
    if found.lower() != declared_commit.lower():
        raise RuntimeError(
            "the checkout at {0}={1!r} is at commit {2}, and {3} declares "
            "{4}. A job that runs a commit nobody asked for RUNS, and the "
            "numbers it returns look exactly like the right ones.".format(
                CLONE_ROOT_ENV, declared_root, found,
                CLONE_COMMIT_ENV, declared_commit))
    return root


ROOT = resolve_repository_root()


# Round-2 design comparison — S1 / S2 / S3, the final metric

Executed report of the second synthetic round the operator commissioned: three
STUDY ARMS derived from Eq. (1), not the production loss. **No arm is the
design until the deliberation publishes r06** — the numbers here are
provisional input to that deliberation.

What is held identical across arms (the control): the round-1 world bitwise
(same generator, same deterministic draw per (scenario, seed)), the same Eq.
(4) cutoffs C̃_k from the same warm-up estimation, and the reliability φ —
the round-1 per-(annotator, category, region) table trained by C3 (Ec. 7)
with the segmentation held fixed. What differs is exactly one thing, the loss
through which the segmentation trains:

- **S1** — no Λ: the gate prunes, survivors weigh 1 (Eq. (1) with Λ replaced
  by the analytic indicator 1{P > C̃_k}) — the thesis' truncation;
- **S2** — gate prunes + Λ weighs: the hard decision is the gate; Λ
  multiplies the survivors' GCE term;
- **S3** — Λ prunes: Eq. (1) exactly as written today.

The segmenter is the declared linear softmax over 2-D class-conditional
Gaussian features, trained in two phases (r05 A3: warm-up first — 300 steps
of plain crowd GCE, identical for every arm — then 1200 steps of the arm's
loss; Adam lr 0.05, full batch, q = 0.7).

The primary metric, never measured before: **consensus segmentation quality**
— mIoU and Dice of argmax P against the synthetic consensus, and the
cross-entropy error of P against the true (point-mass) probabilities. The
secondary metrics are the round-1 four read per arm through its own
mechanism, plus the case-A sensor F_k(1/K) and, for S2 and S3, the mean Λ
over each arm's own survivors per scenario and category.

Re-run headlessly from the repository root:

```
PATH="$PWD/.venv/bin:$PATH" .venv/bin/python -m jupyter nbconvert \
  --to notebook --execute --inplace \
  --ExecutePreprocessor.timeout=7200 notebooks/design_round.ipynb
```


In [2]:
import json
import sys
import time

import torch

sys.path.insert(0, str(ROOT / "src"))

# The fingerprint of the code this report ran against — `report_digest` is the
# single implementation; `verify` recomputes and compares it.
from CrowdSegmenter_CT_GCE import report_digest
print(report_digest.stamp())

from CrowdSegmenter_CT_GCE.design_round import (
    ARMS, DesignConfig, run_design_block)

CONFIG = DesignConfig()   # 30 seeds: the declared verdict floor
print("declared configuration:", json.dumps(CONFIG.as_dict()))
print("arms:", list(ARMS))


SOURCES-SHA256 c609eb535fa3deb66bc39f4305a1a14982a348013a3bb2c53d9cd0735f08de43
declared configuration: {"base": {"n_classes": 4, "n_annotators": 4, "n_regions": 2, "n_pixels": 4000, "coverage": 0.8, "prior": [0.4, 0.3, 0.2, 0.1], "region_split": [0.7, 0.3], "model_error": [0.1, 0.3], "model_eps": 0.15, "lambda_grid": [0.05, 0.2, 0.8, 3.2], "seeds": [0, 1, 2, 3, 4], "branchSteps": 1500, "branchLr": 0.05, "qUnconstrained": 0.7, "collapse": {"saturationFraction": 0.9, "boundary": 0.02, "stdFloor": 0.01}, "pilotSeeds": 5, "verdictFloorSeeds": 30}, "seeds": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29], "thetaWarmupSteps": 300, "thetaArmSteps": 1200, "thetaLr": 0.05, "q": 0.7, "featureMeans": [[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0], [0.0, -1.0]], "lambdaHandover": 0.5}
arms: ['S1', 'S2', 'S3']


## Timing one block before committing the grid

The measured figure decides the scale, never a guess. One (scenario, seed)
block covers the shared half (world, warm-up estimation, the C3 reliability)
and the three arms' segmenter training.

In [3]:
t0 = time.time()
probe = run_design_block(CONFIG, "region-varying", CONFIG.seeds[0])
block_s = time.time() - t0
n_blocks = 3 * len(CONFIG.seeds)
print(f"one block: {block_s:.1f} s | grid: {n_blocks} blocks "
      f"~ {block_s * n_blocks / 60:.0f} min")
print(f"scale: {len(CONFIG.seeds)} seeds per cell — the declared verdict "
      f"floor is 30, so this round aims at the floor")


one block: 16.1 s | grid: 90 blocks ~ 24 min
scale: 30 seeds per cell — the declared verdict floor is 30, so this round aims at the floor


## The grid

Outermost loop: the scenario; then the seed — a repetition varies the material
it draws. The shared half is computed once per (scenario, seed) and handed to
the three arms unchanged.

In [4]:
blocks = []
for scenario in ("region-varying", "per-category", "correlated-bias"):
    for seed in CONFIG.seeds:
        blocks.append(run_design_block(CONFIG, scenario, seed))
        if (len(blocks) % 10) == 0:
            print(f"{len(blocks)}/{n_blocks} blocks done")

out_dir = ROOT / "results" / "selection"
out_dir.mkdir(parents=True, exist_ok=True)
record = {
    "what": "round-2 design comparison -- provisional, feeds the r06 "
            "deliberation; no arm is the design until it publishes",
    "revision": "research-concept-category-truncated-generalized-cross-entropy-for-semantic-se-r10.md",
    "digest": report_digest.stamp(),
    "config": CONFIG.as_dict(),
    "blocks": blocks,
}
(out_dir / "design_round.json").write_text(json.dumps(record, indent=1))
print(f"{len(blocks)} blocks recorded -> {out_dir / 'design_round.json'}")


10/90 blocks done


20/90 blocks done


30/90 blocks done


40/90 blocks done


50/90 blocks done


60/90 blocks done


70/90 blocks done


80/90 blocks done


90/90 blocks done
90 blocks recorded -> /home/lucas/papers/ct-gce-smith/implementations/CrowdSegmenter-CT-GCE/results/selection/design_round.json


## Comparability with round 1

The contract: same world, same warm-up estimation, same C3 reliability per
(scenario, seed). For the seeds the two rounds share (0–4), the shared Λ
metrics must reproduce round 1's C3 rows exactly — same code path, same
draws.

In [5]:
r1_path = ROOT / "results" / "selection" / "selection_pilot.json"
if r1_path.exists():
    r1 = json.loads(r1_path.read_text())
    r1_c3 = {(r["scenario"], r["seed"]): r["metrics"]["recoveryError"]
             for r in r1["runs"] if r["candidate"] == "C3"}
    worst = 0.0
    for block in blocks:
        if block["seed"] > 4:
            continue
        key = (block["scenario"], block["seed"])
        if key in r1_c3:
            worst = max(worst, abs(block["shared"]["lambdaMetrics"]["recoveryError"]
                                   - r1_c3[key]))
    print(f"max |round2 - round1| C3 recovery error over the shared seeds: {worst:.2e}")
    print("comparable" if worst == 0.0 else "NOT BITWISE-COMPARABLE — read the tables with care")
else:
    print("round-1 record not found; the check needs results/selection/selection_pilot.json")


max |round2 - round1| C3 recovery error over the shared seeds: 0.00e+00
comparable


## Primary metric — consensus segmentation quality (mean ± std over 30 seeds)

The metric the two rounds had not measured yet: how good is the consensus the
arm's training signal actually produces. Higher mIoU and Dice are better;
lower cross-entropy is better.

In [6]:
from collections import defaultdict

def mean(xs):
    xs = [x for x in xs if x == x]
    return sum(xs) / len(xs) if xs else float("nan")

def std(xs):
    xs = [x for x in xs if x == x]
    if len(xs) < 2:
        return float("nan")
    m = sum(xs) / len(xs)
    return (sum((x - m) ** 2 for x in xs) / (len(xs) - 1)) ** 0.5

groups = defaultdict(list)
for block in blocks:
    for arm in ARMS:
        groups[(block["scenario"], arm)].append(block["arms"][arm])

SCENARIOS = ("region-varying", "per-category", "correlated-bias")
lines = ["| scenario | arm | mIoU ↑ | Dice ↑ | P cross-entropy ↓ | pruned fraction | poda precision ↑ | poda recall ↑ |",
         "| --- | --- | --- | --- | --- | --- | --- | --- |"]
for scenario in SCENARIOS:
    for arm in ARMS:
        runs = groups[(scenario, arm)]
        miou = [r["primary"]["mIoU"] for r in runs]
        dice = [r["primary"]["Dice"] for r in runs]
        ce = [r["primary"]["pErrorCE"] for r in runs]
        frac = [r["pruning"]["prunedTotal"] / r["pruning"]["annotatedTotal"] for r in runs]
        pp = [r["pruning"]["precision"] for r in runs]
        rc = [r["pruning"]["recall"] for r in runs]
        lines.append("| " + " | ".join([
            scenario, arm,
            f"{mean(miou):.3f} ± {std(miou):.3f}",
            f"{mean(dice):.3f} ± {std(dice):.3f}",
            f"{mean(ce):.3f} ± {std(ce):.3f}",
            f"{mean(frac):.3f}",
            f"{mean(pp):.3f} ± {std(pp):.3f}",
            f"{mean(rc):.3f} ± {std(rc):.3f}",
        ]) + " |")
print("\n".join(lines))


| scenario | arm | mIoU ↑ | Dice ↑ | P cross-entropy ↓ | pruned fraction | poda precision ↑ | poda recall ↑ |
| --- | --- | --- | --- | --- | --- | --- | --- |
| region-varying | S1 | 0.338 ± 0.007 | 0.464 ± 0.007 | 5.909 ± 0.340 | 0.294 | 0.665 ± 0.012 | 0.909 ± 0.006 |
| region-varying | S2 | 0.335 ± 0.008 | 0.461 ± 0.009 | 5.716 ± 0.283 | 0.294 | 0.665 ± 0.012 | 0.909 ± 0.006 |
| region-varying | S3 | 0.369 ± 0.007 | 0.518 ± 0.010 | 1.317 ± 0.050 | 0.107 | 0.612 ± 0.027 | 0.303 ± 0.036 |
| per-category | S1 | 0.304 ± 0.038 | 0.422 ± 0.054 | 6.181 ± 0.671 | 0.319 | 0.652 ± 0.011 | 0.942 ± 0.008 |
| per-category | S2 | 0.291 ± 0.034 | 0.407 ± 0.048 | 6.018 ± 0.612 | 0.319 | 0.652 ± 0.011 | 0.942 ± 0.008 |
| per-category | S3 | 0.313 ± 0.008 | 0.438 ± 0.009 | 1.599 ± 0.041 | 0.089 | 0.574 ± 0.017 | 0.232 ± 0.022 |
| correlated-bias | S1 | 0.339 ± 0.007 | 0.464 ± 0.007 | 6.021 ± 0.333 | 0.338 | 0.691 ± 0.009 | 0.932 ± 0.004 |
| correlated-bias | S2 | 0.333 ± 0.009 | 0.459 ± 0.009 | 5.84

### The shared reliability's own metrics (arm-shared by construction)

The recovery, gate correlation and collapse of the C3-trained Λ are identical
across arms this round — the design holds them fixed so the arms differ in
exactly the training loss. They are reported once per scenario, beside the
truth's own recovery.

In [7]:
lines = ["| scenario | Λ recovery error ↓ | gate correlation | collapse rate |",
         "| --- | --- | --- | --- |"]
for scenario in SCENARIOS:
    sb = [b["shared"] for b in blocks if b["scenario"] == scenario]
    rec = [s["lambdaMetrics"]["recoveryError"] for s in sb]
    gc = [s["lambdaMetrics"]["gateCorrelation"] for s in sb]
    col = [1.0 if s["lambdaMetrics"]["collapse"]["collapsed"] else 0.0 for s in sb]
    lines.append(f"| {scenario} | {mean(rec):.3f} ± {std(rec):.3f} | "
                 f"{mean(gc):.3f} ± {std(gc):.3f} | {mean(col):.2f} |")
print("\n".join(lines))


| scenario | Λ recovery error ↓ | gate correlation | collapse rate |
| --- | --- | --- | --- |
| region-varying | 0.034 ± 0.004 | 0.383 ± 0.011 | 0.00 |
| per-category | 0.045 ± 0.006 | 0.260 ± 0.026 | 0.00 |
| correlated-bias | 0.043 ± 0.006 | 0.234 ± 0.014 | 0.00 |


## The case-A sensor — F_k(1/K) per category after the warm-up

Round-1 definition, world-level, arm-shared: where F_k(1/K) exceeds ρ̂_k the
1/K clip discards more than the noise estimate asks, and the difference is
the over-discarding, by category.

In [8]:
lines = ["| scenario | category | ρ̂_k | C̃_k | F_k(1/K) | over-discarding |",
         "| --- | --- | --- | --- | --- | --- |"]
for scenario in SCENARIOS:
    per_seed = [b["shared"]["sensor"] for b in blocks if b["scenario"] == scenario]
    k = len(per_seed[0]["fFloor"])
    for cat in range(k):
        rho = mean([s["rhoHat"][cat] for s in per_seed])
        cut = mean([s["cutoffs"][cat] for s in per_seed])
        ff = mean([s["fFloor"][cat] for s in per_seed])
        over = mean([s["overDiscard"][cat] for s in per_seed])
        lines.append(f"| {scenario} | {cat} | {rho:.3f} | {cut:.3f} | {ff:.3f} | {over:+.3f} |")
print("\n".join(lines))


| scenario | category | ρ̂_k | C̃_k | F_k(1/K) | over-discarding |
| --- | --- | --- | --- | --- | --- |
| region-varying | 0 | 0.119 | 0.250 | 0.220 | +0.101 |
| region-varying | 1 | 0.171 | 0.250 | 0.261 | +0.090 |
| region-varying | 2 | 0.248 | 0.250 | 0.337 | +0.089 |
| region-varying | 3 | 0.405 | 0.250 | 0.480 | +0.076 |
| per-category | 0 | 0.148 | 0.250 | 0.259 | +0.111 |
| per-category | 1 | 0.180 | 0.250 | 0.281 | +0.100 |
| per-category | 2 | 0.290 | 0.332 | 0.368 | +0.078 |
| per-category | 3 | 0.576 | 0.463 | 0.594 | +0.018 |
| correlated-bias | 0 | 0.155 | 0.250 | 0.253 | +0.098 |
| correlated-bias | 1 | 0.199 | 0.250 | 0.299 | +0.100 |
| correlated-bias | 2 | 0.253 | 0.250 | 0.387 | +0.134 |
| correlated-bias | 3 | 0.370 | 0.250 | 0.536 | +0.167 |


## The mean Λ over surviving annotations (S2 and S3, per category)

The statistic that will later decide whether the weight needs normalization:
S2's survivors are the gate's, S3's are the Λ ≥ 0.5 handover set. A category
whose survivors carry a low mean Λ is a category whose training signal the
arm under-weights.

In [9]:
lines = ["| scenario | arm | cat 0 | cat 1 | cat 2 | cat 3 |",
         "| --- | --- | --- | --- | --- | --- |"]
for scenario in SCENARIOS:
    for arm in ("S2", "S3"):
        per_seed = [b["arms"][arm]["survivingLambda"] for b in blocks
                    if b["scenario"] == scenario]
        k = len(per_seed[0])
        means = [mean([s[cat] for s in per_seed]) for cat in range(k)]
        lines.append(f"| {scenario} | {arm} | " +
                     " | ".join(f"{m:.3f}" for m in means) + " |")
print("\n".join(lines))


| scenario | arm | cat 0 | cat 1 | cat 2 | cat 3 |
| --- | --- | --- | --- | --- | --- |
| region-varying | S2 | 0.901 | 0.863 | 0.791 | 0.641 |
| region-varying | S3 | 0.879 | 0.838 | 0.827 | 0.725 |
| per-category | S2 | 0.852 | 0.826 | 0.721 | 0.435 |
| per-category | S3 | 0.848 | 0.822 | 0.716 | 0.529 |
| correlated-bias | S2 | 0.859 | 0.803 | 0.695 | 0.506 |
| correlated-bias | S3 | 0.855 | 0.799 | 0.692 | 0.533 |


## What the round separates, and what it leaves to the deliberation

Generated from the record: two arms separate on a metric when their means
differ by more than the combined standard error over the 30 seeds. **No arm
is the design** — the design decision is the deliberation's (r06); where the
margins do not separate, this report says so plainly.

In [10]:
import math

def se(xs):
    s = std(xs)
    n = len([x for x in xs if x == x])
    return s / math.sqrt(n) if n and s == s else float("nan")

PRIMARY = [("primary", "mIoU", "mIoU", "higher"),
           ("primary", "Dice", "Dice", "higher"),
           ("primary", "pErrorCE", "P cross-entropy", "lower"),
           ("pruning", "precision", "poda precision", "higher"),
           ("pruning", "recall", "poda recall", "higher")]

def arm_values(scenario, arm, section, key):
    return [b["arms"][arm][section][key] for b in blocks
            if b["scenario"] == scenario]

print("Separation diagnostics over", len(CONFIG.seeds), "seeds per cell:")
for scenario in SCENARIOS:
    for section, key, label, direction in PRIMARY:
        pairs = []
        for i in range(len(ARMS)):
            for j in range(i + 1, len(ARMS)):
                a = arm_values(scenario, ARMS[i], section, key)
                b = arm_values(scenario, ARMS[j], section, key)
                s1, s2 = se(a), se(b)
                if s1 != s1 or s2 != s2:
                    continue
                gap = abs(mean(a) - mean(b))
                combined = math.sqrt(s1 ** 2 + s2 ** 2)
                if combined > 0 and gap > combined:
                    better = ARMS[i] if ((mean(a) < mean(b)) if direction == "lower"
                                         else (mean(a) > mean(b))) else ARMS[j]
                    pairs.append(f"{ARMS[i]} vs {ARMS[j]} -> {better} "
                                 f"(gap {gap:.3f} > SE {combined:.3f})")
        if pairs:
            print(f"- {scenario} / {label}: SEPARATES: " + "; ".join(pairs))
        else:
            print(f"- {scenario} / {label}: indistinguishable at "
                  f"{len(CONFIG.seeds)} seeds")
print()
print("NO ARM IS THE DESIGN: the tables above are provisional evidence for "
      "the r06 deliberation, which owns the design decision.")


Separation diagnostics over 30 seeds per cell:
- region-varying / mIoU: SEPARATES: S1 vs S2 -> S1 (gap 0.003 > SE 0.002); S1 vs S3 -> S3 (gap 0.031 > SE 0.002); S2 vs S3 -> S3 (gap 0.034 > SE 0.002)
- region-varying / Dice: SEPARATES: S1 vs S2 -> S1 (gap 0.003 > SE 0.002); S1 vs S3 -> S3 (gap 0.054 > SE 0.002); S2 vs S3 -> S3 (gap 0.058 > SE 0.002)
- region-varying / P cross-entropy: SEPARATES: S1 vs S2 -> S2 (gap 0.192 > SE 0.081); S1 vs S3 -> S3 (gap 4.592 > SE 0.063); S2 vs S3 -> S3 (gap 4.400 > SE 0.052)
- region-varying / poda precision: SEPARATES: S1 vs S3 -> S1 (gap 0.053 > SE 0.005); S2 vs S3 -> S2 (gap 0.053 > SE 0.005)
- region-varying / poda recall: SEPARATES: S1 vs S3 -> S1 (gap 0.606 > SE 0.007); S2 vs S3 -> S2 (gap 0.606 > SE 0.007)
- per-category / mIoU: SEPARATES: S1 vs S2 -> S1 (gap 0.013 > SE 0.009); S1 vs S3 -> S3 (gap 0.009 > SE 0.007); S2 vs S3 -> S3 (gap 0.022 > SE 0.006)
- per-category / Dice: SEPARATES: S1 vs S2 -> S1 (gap 0.015 > SE 0.013); S1 vs S3 -> S3 (gap 